Import Library

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import cv2
from scipy.fftpack import dct
import matplotlib.pyplot as plt
from PIL import Image
import os
import kagglehub
from google.colab import files

# Cek apakah GPU aktif
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Siap! Menggunakan device: {device}")

✅ Siap! Menggunakan device: cpu


Download Dataset

In [3]:
print("⬇️ Sedang mendownload dataset dari Kaggle...")
# Download dataset
dataset_path = kagglehub.dataset_download("aryankaushik005/custom-dataset")

print(f"✅ Dataset selesai didownload di: {dataset_path}")

# Logika otomatis mencari folder 'train' yang berisi subfolder kelas
target_dir = dataset_path
found_train = False

# Mencari folder yang mungkin bernama 'train', 'Train', atau langsung berisi kelas
possible_subfolders = ['train', 'Train']
for sub in possible_subfolders:
    if sub in os.listdir(dataset_path):
        target_dir = os.path.join(dataset_path, sub)
        found_train = True
        break

print(f"📂 Target folder dataset: {target_dir}")
print(f"   Isi folder: {os.listdir(target_dir)}")

⬇️ Sedang mendownload dataset dari Kaggle...


100%|██████████| 4.76G/4.76G [00:57<00:00, 89.3MB/s]

Extracting files...


✅ Dataset selesai didownload di: /root/.cache/kagglehub/datasets/aryankaushik005/custom-dataset/versions/1
📂 Target folder dataset: /root/.cache/kagglehub/datasets/aryankaushik005/custom-dataset/versions/1
   Isi folder: ['content']


Definisi Kelas "Novelty" (Analisis Krominansi)

In [4]:
class ChrominanceFrequencyTransform:
    """
    Novelty: Mengubah gambar RGB menjadi representasi spektrum frekuensi
    berbasis warna (YCbCr) untuk mendeteksi jejak deepfake.
    """
    def __call__(self, img):
        img_np = np.array(img)

        # Handle jika input grayscale
        if len(img_np.shape) < 3:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)

        # 1. Konversi ke YCrCb (Fokus ke warna, bukan gelap-terang)
        img_ycc = cv2.cvtColor(img_np, cv2.COLOR_RGB2YCrCb)

        # 2. Ambil channel Cr (Merah) dan Cb (Biru)
        chroma_1 = img_ycc[:,:,1]
        chroma_2 = img_ycc[:,:,2]

        # 3. Lakukan DCT (Discrete Cosine Transform)
        dct_1 = dct(dct(chroma_1.T, norm='ortho').T, norm='ortho')
        dct_2 = dct(dct(chroma_2.T, norm='ortho').T, norm='ortho')

        # 4. Log Transform (Agar pola terlihat jelas)
        dct_log_1 = np.log(np.abs(dct_1) + 1e-12)
        dct_log_2 = np.log(np.abs(dct_2) + 1e-12)

        # 5. Gabungkan rata-rata
        dct_combined = (dct_log_1 + dct_log_2) / 2

        # 6. Normalisasi 0-1
        dct_norm = (dct_combined - np.min(dct_combined)) / (np.max(dct_combined) - np.min(dct_combined))

        # 7. Stack menjadi 3 channel agar kompatibel dengan ResNet
        dct_stacked = np.stack([dct_norm] * 3, axis=2)

        return dct_stacked.astype(np.float32) # Return format yang siap untuk PyTorch

    # Helper function untuk visualisasi nanti
    def get_visual(self, img):
        tensor_output = self.__call__(img)
        return tensor_output[:,:,0] # Ambil 1 layer saja untuk di-plot

Persiapan Model & Training Loop

In [ ]:
# Konfigurasi Pipeline
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    ChrominanceFrequencyTransform(), # Panggil Novelty
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load Data
try:
    train_data = datasets.ImageFolder(target_dir, transform=data_transforms)
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    class_names = train_data.classes
    print(f"✅ Kelas ditemukan: {class_names}")
    # Simpan urutan kelas untuk testing
    label_map = {i: name for i, name in enumerate(class_names)}
except Exception as e:
    print(f"❌ Error loading data: {e}")

# Load Model ResNet18 Pretrained
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) # Ganti output jadi 2 kelas
model = model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# --- Mulai Training ---
num_epochs = 3
print("\n Mulai Training...")

model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Print progress bar sederhana
        if i % 20 == 0:
            print(f".", end="")

    epoch_acc = 100 * correct / total
    print(f"\nEpoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(train_loader):.4f} | Akurasi: {epoch_acc:.2f}%")

print("\n✅ Training Selesai!")
torch.save(model.state_dict(), "my_deepfake_model.pth")
print("💾 Model disimpan sebagai 'my_deepfake_model.pth'")
print("⬇️ Mendownload file .pth ke lokal...")
files.download('my_deepfake_model.pth')

✅ Kelas ditemukan: ['content']

 Mulai Training...
................................................................................................................................................................................................